In [ ]:
import sys
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import skimage as ski

from neural_reconstruction.core.topology import TopologyBuilder
from neural_reconstruction.core.preprocessing import dilate_epidermis_vertically

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
DINOV3_REPO = '/home/pony/projects/ienf_q/dinov3'
CKPT_PATH   = '/home/pony/projects/ienf_q/dino/dinov3_convnext_tiny_pretrain_lvd1689m-21b726bb.pth'
DATA_DIR    = Path('/home/pony/projects/ienf_q/data_0331')
PATCH_SIZE  = 128   # px, covers both seeds + context

sys.path.insert(0, DINOV3_REPO)
print(f'Device: {DEVICE}')

## Backbone — DINOv3 ConvNeXt-Tiny, first conv adapted to 2 channels

In [ ]:
def load_2ch_backbone(repo: str, ckpt: str) -> nn.Module:
    """
    Load DINOv3 ConvNeXt-Tiny and adapt the first Conv2d from 3→2 input channels.

    Weight init for 2-channel conv:
      Ch0 (image) : mean of original R+G+B weights
      Ch1 (mask)  : same mean  (will adapt quickly during fine-tuning)
    """
    m = torch.hub.load(repo, 'dinov3_convnext_tiny', source='local', weights=ckpt)

    # Locate first Conv2d  (downsample_layers[0][0])
    first_conv = m.downsample_layers[0][0]
    assert isinstance(first_conv, nn.Conv2d) and first_conv.in_channels == 3

    old_w = first_conv.weight.data          # (96, 3, 4, 4)
    mean_w = old_w.mean(dim=1, keepdim=True)  # (96, 1, 4, 4)
    new_w  = mean_w.expand(-1, 2, -1, -1).clone()  # (96, 2, 4, 4)

    new_conv = nn.Conv2d(
        2, first_conv.out_channels,
        kernel_size=first_conv.kernel_size,
        stride=first_conv.stride,
        padding=first_conv.padding,
        bias=first_conv.bias is not None,
    )
    new_conv.weight.data = new_w
    if first_conv.bias is not None:
        new_conv.bias.data = first_conv.bias.data.clone()

    m.downsample_layers[0][0] = new_conv
    print(f'First conv replaced: (3, ...) → (2, ...)')
    return m


backbone = load_2ch_backbone(DINOV3_REPO, CKPT_PATH).to(DEVICE)

# Sanity check
with torch.no_grad():
    feat = backbone(torch.zeros(2, 2, PATCH_SIZE, PATCH_SIZE, device=DEVICE))
print(f'Backbone output: {feat.shape}')   # (2, 768)
FEAT_DIM = feat.shape[-1]

## Data pipeline

For each cross-fragment seed pair `(A, B)` in the same weka sample:
- Crop a `PATCH_SIZE × PATCH_SIZE` window centred at the midpoint of A and B
- **Ch0**: image (grayscale, normalised)
- **Ch1**: combined binary mask — annotation component A ∪ component B
- **Label**: 1 if A and B share the same `label_comp_id`, else 0

In [ ]:
# Grayscale normalisation stats (ImageNet green-channel approx.)
_IMG_MEAN = 0.449
_IMG_STD  = 0.226


def extract_2ch_patch(
    image_gray: np.ndarray,
    annot_comp:  np.ndarray,
    seed_a: tuple[int, int],
    seed_b: tuple[int, int],
    size: int,
) -> np.ndarray:
    """
    Returns float32 array of shape (2, size, size):
      [0] image patch, normalised
      [1] binary mask of annotation_comp_A | annotation_comp_B, float {0, 1}
    Regions outside image bounds are zero-padded.
    """
    H, W = image_gray.shape
    cy = (seed_a[0] + seed_b[0]) // 2
    cx = (seed_a[1] + seed_b[1]) // 2
    half = size // 2

    r0, r1 = cy - half, cy - half + size
    c0, c1 = cx - half, cx - half + size
    sr0, sr1 = max(r0, 0), min(r1, H)
    sc0, sc1 = max(c0, 0), min(c1, W)
    dr0, dr1 = sr0 - r0, sr1 - r0
    dc0, dc1 = sc0 - c0, sc1 - c0

    # Channel 0: image
    ch_img  = np.zeros((size, size), dtype=np.float32)
    ch_img[dr0:dr1, dc0:dc1] = image_gray[sr0:sr1, sc0:sc1].astype(np.float32) / 255.0
    ch_img = (ch_img - _IMG_MEAN) / _IMG_STD

    # Channel 1: combined annotation mask
    comp_a = annot_comp[seed_a[0], seed_a[1]]
    comp_b = annot_comp[seed_b[0], seed_b[1]]
    mask_full = ((annot_comp == comp_a) | (annot_comp == comp_b)).astype(np.float32)
    ch_mask = np.zeros((size, size), dtype=np.float32)
    ch_mask[dr0:dr1, dc0:dc1] = mask_full[sr0:sr1, sc0:sc1]

    return np.stack([ch_img, ch_mask], axis=0)   # (2, H, W)


def build_sample_pairs(sample_dir: Path, patch_size: int = PATCH_SIZE) -> list[dict]:
    """
    Build all cross-fragment seed pairs for one sample.
    Returns list of dicts with keys: patch (2,P,P), label (int), sample_id.
    """
    for fname in ('image.png', 'weka.png', 'label.png', 'mask.png'):
        if not (sample_dir / fname).exists():
            return []

    image_rgb = cv2.imread(str(sample_dir / 'image.png'), cv2.IMREAD_COLOR_RGB)
    image_g   = image_rgb[:, :, 1]
    mask      = cv2.imread(str(sample_dir / 'mask.png'),  cv2.IMREAD_GRAYSCALE)
    weka      = cv2.imread(str(sample_dir / 'weka.png'),  cv2.IMREAD_GRAYSCALE)
    label_img = cv2.imread(str(sample_dir / 'label.png'), cv2.IMREAD_GRAYSCALE)

    roi_mask = dilate_epidermis_vertically(mask, offset_px=50)
    bg       = cv2.morphologyEx(image_g, cv2.MORPH_OPEN,
                                cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (51, 51)))
    image_g  = cv2.subtract(image_g, bg)

    roi_img   = cv2.bitwise_and(image_g,   image_g,   mask=roi_mask)
    roi_weka  = cv2.bitwise_and(weka,      weka,      mask=roi_mask)
    roi_label = cv2.bitwise_and(label_img, label_img, mask=roi_mask)

    k3 = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    roi_weka  = cv2.morphologyEx(roi_weka,  cv2.MORPH_CLOSE, k3, iterations=3)
    roi_label = cv2.morphologyEx(roi_label, cv2.MORPH_CLOSE, k3, iterations=3)
    roi_weka[roi_weka > 0]   = 255
    roi_label[roi_label > 0] = 255

    annot_comp = ski.measure.label(roi_weka  > 0, connectivity=2).astype(int)
    label_comp = ski.measure.label(roi_label > 0, connectivity=2).astype(int)

    try:
        tb    = TopologyBuilder(segment_length=10.0)
        graph = tb.build_seed_graph(roi_weka, roi_img)
    except Exception:
        return []

    # Collect per-seed metadata
    seeds = []
    for (y, x) in graph.nodes():
        y, x = int(y), int(x)
        seeds.append({
            'yx':          (y, x),
            'annot_comp':  int(annot_comp[y, x]),
            'label_comp':  int(label_comp[y, x]),
        })

    # Build pairs: cross-fragment only (different annot_comp)
    records = []
    n = len(seeds)
    for i in range(n):
        for j in range(i + 1, n):
            sa, sb = seeds[i], seeds[j]
            if sa['annot_comp'] == sb['annot_comp']:
                continue   # same fragment — skip
            # if sa['label_comp'] == 0 or sb['label_comp'] == 0:
            #     continue   # not on GT label — skip

            # Distance filter: skip if too far apart to fit in patch
            dy = sa['yx'][0] - sb['yx'][0]
            dx = sa['yx'][1] - sb['yx'][1]
            dist = (dy**2 + dx**2) ** 0.5
            if dist > patch_size * 0.8:
                continue

            patch = extract_2ch_patch(
                roi_img, annot_comp,
                sa['yx'], sb['yx'],
                patch_size,
            )
            label = int(sa['label_comp'] == sb['label_comp'])
            records.append({
                'patch':     patch,
                'label':     label,
                'sample_id': sample_dir.name,
            })
    return records


print('Patch utilities ready.')

In [ ]:
all_records = []
sample_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])

for sd in tqdm(sample_dirs, desc='Building pairs'):
    all_records.extend(build_sample_pairs(sd))

n_pos = sum(r['label'] for r in all_records)
n_neg = len(all_records) - n_pos
print(f'Total pairs: {len(all_records)}  (pos={n_pos}, neg={n_neg}, ratio={n_pos/max(1,len(all_records)):.2f})')

## Dataset

In [ ]:
class PairDataset(Dataset):
    def __init__(self, records: list[dict], oversample_pos: bool = True):
        pos = [r for r in records if r['label'] == 1]
        neg = [r for r in records if r['label'] == 0]

        if oversample_pos and len(pos) < len(neg):
            rng    = np.random.default_rng(0)
            idx    = rng.integers(len(pos), size=len(neg) - len(pos))
            pos    = pos + [pos[i] for i in idx]

        self.records = pos + neg
        print(f'  Dataset: {len(pos)} pos + {len(neg)} neg = {len(self.records)}')

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        return (
            torch.from_numpy(r['patch']),                       # (2, P, P)
            torch.tensor(r['label'], dtype=torch.float32),
        )


# 80/20 split by sample_id
all_ids   = sorted({r['sample_id'] for r in all_records})
split     = int(len(all_ids) * 0.8)
train_ids = set(all_ids[:split])
val_ids   = set(all_ids[split:])

train_recs = [r for r in all_records if r['sample_id'] in train_ids]
val_recs   = [r for r in all_records if r['sample_id'] in val_ids]
print(f'Train samples: {len(train_ids)}, Val samples: {len(val_ids)}')

train_ds = PairDataset(train_recs, oversample_pos=True)
val_ds   = PairDataset(val_recs,   oversample_pos=False)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

## Model — fine-tuned DINOv3 backbone + FC head

```
(2, P, P)  ──►  DINOv3 ConvNeXt-Tiny  ──►  (768,)  ──►  FC head  ──►  logit
```

In [ ]:
class PairClassifier(nn.Module):
    def __init__(self, backbone: nn.Module, feat_dim: int):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(x)      # (B, feat_dim)
        return self.head(feat).squeeze(-1)  # (B,)


model = PairClassifier(backbone, feat_dim=FEAT_DIM).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} params')

## Training

In [ ]:
NUM_EPOCHS = 20

# Differential LR: backbone gets 10× lower LR than head
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-5},
    {'params': model.head.parameters(),     'lr': 1e-4},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.BCEWithLogitsLoss()


def run_epoch(loader, train: bool):
    model.train(train)
    total_loss = correct = total = 0
    ctx = torch.enable_grad if train else torch.no_grad
    with ctx():
        for patches, labels in loader:
            patches, labels = patches.to(DEVICE), labels.to(DEVICE)
            logits = model(patches)
            loss   = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            correct    += ((logits.detach().sigmoid() > 0.5) == labels.bool()).sum().item()
            total      += len(labels)
    return total_loss / total, correct / total


history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader,   train=False)
    scheduler.step()
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)
    print(f'Epoch {epoch:3d}/{NUM_EPOCHS}  '
          f'train loss={tr_loss:.4f} acc={tr_acc*100:.1f}%  '
          f'val loss={va_loss:.4f} acc={va_acc*100:.1f}%')

torch.save({'backbone': model.backbone.state_dict(),
            'head':     model.head.state_dict()}, 'marker_classifier.pth')
print('Saved.')

## Evaluation

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report

model.eval()
all_logits, all_labels = [], []
with torch.no_grad():
    for patches, labels in val_loader:
        all_logits.append(model(patches.to(DEVICE)).cpu())
        all_labels.append(labels) 
 
logits = torch.cat(all_logits).numpy()
labels = torch.cat(all_labels).numpy().astype(int)
probs  = 1 / (1 + np.exp(-logits))
preds  = (probs > 0.5).astype(int)

print(f'Val AUC: {roc_auc_score(labels, probs):.4f}')
print(classification_report(labels, preds, target_names=['different', 'same']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, NUM_EPOCHS + 1)
axes[0].plot(ep, history['train_loss'], label='train')
axes[0].plot(ep, history['val_loss'],   label='val')
axes[0].set(xlabel='Epoch', ylabel='BCE Loss', title='Loss'); axes[0].legend()
axes[1].plot(ep, [a*100 for a in history['train_acc']], label='train')
axes[1].plot(ep, [a*100 for a in history['val_acc']],   label='val')
axes[1].set(xlabel='Epoch', ylabel='Accuracy (%)', title='Accuracy'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Visualise sample pairs — show Ch0 (image) and Ch1 (mask) side by side
categories = {
    'TP (same  → pred same)':  (labels == 1) & (preds == 1),
    'TN (diff  → pred diff)':  (labels == 0) & (preds == 0),
    'FP (diff  → pred same)':  (labels == 0) & (preds == 1),
    'FN (same  → pred diff)':  (labels == 1) & (preds == 0),
}
N_SHOW = 3
fig, axes = plt.subplots(len(categories), N_SHOW * 2, figsize=(N_SHOW * 5, len(categories) * 2.5))

val_patches = [val_ds[i][0] for i in range(len(val_ds))]

for row, (cat, mask) in enumerate(categories.items()):
    idxs = np.where(mask)[0][:N_SHOW]
    for col, idx in enumerate(idxs):
        p = val_patches[idx].numpy()    # (2, P, P)
        axes[row][col*2].imshow(p[0], cmap='gray')
        axes[row][col*2].set_title(f'img  p={probs[idx]:.2f}', fontsize=8)
        axes[row][col*2].axis('off')
        axes[row][col*2+1].imshow(p[1], cmap='hot')
        axes[row][col*2+1].set_title('mask', fontsize=8)
        axes[row][col*2+1].axis('off')
    axes[row][0].set_ylabel(cat, fontsize=8)

plt.suptitle('Validation pairs  (left=image patch, right=annotation mask)', fontsize=10)
plt.tight_layout(); plt.show()